# Bank Customer Churn Prediction

Predict which bank customers are likely to **leave (churn)** from their profile and account activity.

**Data:** 10,000 customers from France, Germany, and Spain. About **20% churned**. This is a **binary classification** problem, and the classes are imbalanced (80/20), so accuracy alone won't tell the real story.

**Approach**
1. Load data and check data quality
2. Explore the data — univariate, bivariate, correlations — and write down what each view tells us
3. Split the data (train / validation / test) and build a preprocessing pipeline
4. Baseline model (no tuning) as a reference point
5. Engineer a couple of EDA-motivated features and confirm with cross-validation that they help
6. Compare several models fairly (same CV folds, same metric)
7. Tune the strongest candidate
8. Pick a threshold two ways — maximum F1, and maximum net benefit under *assumed* costs — and choose between them
9. Check for overfitting and for calibration (is the score a real probability?)
10. Evaluate once on a held-out test set, with bootstrap confidence intervals
11. Explain the model with feature importance and SHAP
12. Save the model, and export scored customers and metrics for the Power BI dashboard

**Reading the numbers:** every result in this notebook is produced by running it. Nothing is typed in by hand, so if you change the data or the settings in the config cell, re-run everything.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_validate,
                                     cross_val_predict, GridSearchCV)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve,
    classification_report, confusion_matrix,
)
import joblib

# One place for every setting used later — change here, not throughout the notebook
RANDOM_STATE = 42
CSV_PATH = "../data/churn_raw.csv"      # run this notebook from the notebooks/ folder
TARGET_COL = "churn"                 # name AFTER the rename in section 2
OUT_DIR = "../dashboard/data"        # where the dashboard CSVs are written (section 19)

SELECT_METRIC = "roc_auc"            # metric used to pick and tune the model (threshold-free)
USE_ENGINEERED_FEATURES = True       # set False to train without zero_balance / age_group
THRESHOLD_RULE = "f1"                # "f1" = maximise F1 on validation, "cost" = maximise net benefit

# ---- ASSUMPTIONS (not facts) -------------------------------------------------------------
# The dataset has no cost information. These three numbers are placeholders chosen to show how
# a cost-based threshold works. Replace them with your bank's real figures before trusting
# anything that depends on them (cost_threshold, net_benefit.csv, the dashboard's net benefit).
OFFER_COST = 1        # cost of one retention offer, in "units"
VALUE_PER_SAVE = 10   # value of keeping one customer who would have left, same units
SUCCESS_RATE = 0.30   # share of contacted churners the offer actually keeps
# ------------------------------------------------------------------------------------------

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["figure.figsize"] = (8, 4)

## 2. Load the data

In [ ]:
raw = pd.read_csv(CSV_PATH)

# This dataset circulates with two sets of column names (e.g. "CustomerId" / "NumOfProducts" or
# "customer_id" / "products_number"). The second set is mapped onto the first, so the notebook runs
# on either file. If the file already uses the first set, this step changes nothing.
ALT_NAMES = {
    "customer_id": "CustomerId", "credit_score": "CreditScore", "country": "Geography",
    "gender": "Gender", "age": "Age", "tenure": "Tenure", "balance": "Balance",
    "products_number": "NumOfProducts", "credit_card": "HasCrCard",
    "active_member": "IsActiveMember", "estimated_salary": "EstimatedSalary", "churn": "Exited",
}
raw = raw.rename(columns=ALT_NAMES)
if "Surname" not in raw.columns:       # some versions have no surname / row-number column
    raw["Surname"] = ""                # blank, so the dashboard export keeps the same columns
if "RowNumber" not in raw.columns:
    raw["RowNumber"] = range(1, len(raw) + 1)

# Keep the identifiers aside (needed later to build the call list), but never train on them
ids = raw[["CustomerId", "Surname"]].copy()

# Rename to clean, consistent lowercase names
RENAME = {
    "CreditScore": "credit_score", "Geography": "country", "Gender": "gender",
    "Age": "age", "Tenure": "tenure", "Balance": "balance",
    "NumOfProducts": "products", "HasCrCard": "credit_card",
    "IsActiveMember": "active", "EstimatedSalary": "salary", "Exited": "churn",
}
df = raw.drop(columns=["RowNumber", "CustomerId", "Surname"]).rename(columns=RENAME)

print("Shape:", df.shape)
df.head()

**Columns**

| Column | Meaning |
|---|---|
| credit_score | Customer's credit score |
| country | France / Germany / Spain |
| gender | Male / Female |
| age | Age in years |
| tenure | Years as a customer |
| balance | Account balance |
| products | Number of bank products held |
| credit_card | Has a credit card (0/1) |
| active | Active member (0/1) |
| salary | Estimated annual salary |
| churn | Target — left the bank (1) or stayed (0) |

## 3. Data quality checks

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print()
print("Churn rate:", round(df["churn"].mean(), 4))

**Inference — data quality**
No missing values, no duplicates — the dataset is clean. About 1 in 5 customers churned, confirming this is an imbalanced classification problem: a model that always predicts "stays" would already be ~80% accurate without being useful, so accuracy alone is the wrong metric to optimize.

## 4. Univariate analysis

In [ ]:
num_cols = ["credit_score", "age", "tenure", "balance", "salary"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(col)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

df[num_cols].skew()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.countplot(x="country", data=df, ax=axes[0])
sns.countplot(x="products", data=df, ax=axes[1])
sns.countplot(x="churn", data=df, ax=axes[2])
axes[0].set_title("Country")
axes[1].set_title("Products held")
axes[2].set_title("Churn (0=stay, 1=leave)")
plt.tight_layout()
plt.show()

**Inference — univariate**
- `balance` has a large spike at 0 (customers with no balance) — worth flagging as its own feature rather than letting the model treat it as "just a low number."
- `age` is right-skewed with most customers between 30–45.
- Most customers hold 1–2 products; very few hold 3–4.

### 4.1 Outlier check — should any values be capped or removed?

An **outlier** is a value far from the rest of its column. Being unusual is not the same as being wrong, so the question is not "does one exist?" but "should we treat it?". Two standard rules flag candidates:

- **z-score** — how many standard deviations a value sits from the column mean. |z| > 3 is the usual flag.
- **IQR (box-plot) rule** — a value more than 1.5 × IQR beyond the quartiles is a **mild** outlier; more than 3 × IQR beyond is an **extreme** outlier.

We would remove or cap a value only if it is **impossible** (a data error) or if it **distorts the model**. The next cells check the first reason; section 10.1 checks the second.

In [ ]:
# Flag outliers with both rules, and grade them mild vs extreme (IQR rule)
rows = []
for col in num_cols:
    s = df[col]
    z = (s - s.mean()) / s.std()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    beyond_1_5 = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)     # mild OR extreme
    beyond_3   = (s < q1 - 3.0 * iqr) | (s > q3 + 3.0 * iqr)     # extreme only
    rows.append({
        "column": col,
        "skew": s.skew(),
        "min": s.min(), "max": s.max(),
        "max |z|": z.abs().max(),
        "|z| > 3": int((z.abs() > 3).sum()),
        "mild (IQR)": int((beyond_1_5 & ~beyond_3).sum()),
        "extreme (IQR)": int(beyond_3.sum()),
        "% of rows flagged": 100 * beyond_1_5.mean(),
    })
outlier_table = pd.DataFrame(rows).set_index("column")
outlier_table


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, col in zip(axes, num_cols):
    sns.boxplot(y=df[col], ax=ax, color="lightsteelblue", flierprops={"markersize": 3})
    ax.set_title(col)
    ax.set_ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
# Do the flagged customers behave differently?  Compare their churn rate with everyone else's.
from scipy.stats import binomtest

def beyond(s, k):                     # True where a value lies more than k x IQR outside the quartiles
    q1, q3 = s.quantile([0.25, 0.75])
    return (s < q1 - k * (q3 - q1)) | (s > q3 + k * (q3 - q1))

base_rate = df[TARGET_COL].mean()
groups = [("credit_score", "mild",    "credit_score", beyond(df["credit_score"], 1.5) & ~beyond(df["credit_score"], 3)),
          ("age",          "mild",    "age",          beyond(df["age"], 1.5) & ~beyond(df["age"], 3)),
          ("age",          "extreme", "age",          beyond(df["age"], 3))]

rows = []
for label, grade, col, flag in groups:
    k, n = int(df.loc[flag, TARGET_COL].sum()), int(flag.sum())
    ci = binomtest(k, n, base_rate).proportion_ci(method="wilson")          # 95% interval for their churn rate
    rows.append({"column": label, "grade": grade, "customers": n,
                 "values range": f"{df.loc[flag, col].min()} to {df.loc[flag, col].max()}",
                 "churn rate (flagged)": k / n,
                 "95% interval": f"{ci.low:.2f} to {ci.high:.2f}"})

print(f"Overall churn rate: {base_rate:.3f}")
pd.DataFrame(rows)


**Inference — outliers**
- Only `age` and `credit_score` have flagged values. `tenure`, `balance` and `salary` have none (the real quirk in `balance` is the spike at 0, already handled by the `zero_balance` feature).
- **`age`:** 347 of the 359 flagged customers are *mild* outliers (63–80), the long right tail behind the skew of about 1. Only 12 are *extreme* (81–92), and the largest |z| is about 5. A 92-year-old customer is rare but entirely possible, so these are real customers, not data errors. The mild group churns at about the overall rate (its interval contains 0.20), so it is not distorting the picture; 12 extreme customers are too few to say anything either way.
- **`credit_score`:** 15 low scores (350–382), all *mild* and all inside the real 350–850 range. All 15 churned (interval 0.80 to 1.00, far above the overall 0.20), so these values **carry signal**; capping them up to the fence (383) would blur it. It is a small group, so treat it as a hint rather than a rule.
- **Decision: keep every value.** None is impossible, nearly all are mild, and the one group that behaves differently behaves *informatively*. Section 10.1 tests the other reason to treat outliers: whether capping improves the model.


## 5. Bivariate analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.barplot(x="country", y="churn", data=df, ax=axes[0, 0])
axes[0, 0].set_title("Churn rate by country")

sns.barplot(x="active", y="churn", data=df, ax=axes[0, 1])
axes[0, 1].set_title("Churn rate by active membership")

sns.barplot(x="products", y="churn", data=df, ax=axes[1, 0])
axes[1, 0].set_title("Churn rate by products held")

age_bins = pd.cut(df["age"], bins=[0, 30, 40, 50, 60, 100])
sns.barplot(x=age_bins, y=df["churn"], ax=axes[1, 1])
axes[1, 1].set_title("Churn rate by age group")
axes[1, 1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

**Inference — bivariate**
- **Germany** churns noticeably more than France or Spain.
- **Inactive** members churn far more than active ones — this is one of the strongest single signals in the data.
- Customers with **3–4 products** churn at a very high rate (likely a sign of a frustrated customer, not a loyal one) while **2 products** is the safest spot.
- Churn rises sharply after **age 40** — a step change, not a straight line, which is worth encoding as its own feature rather than leaving age as a raw number.

## 6. Correlation

In [ ]:
num_cols_all = ["credit_score", "age", "tenure", "balance", "products", "salary", "churn"]
plt.figure(figsize=(7, 5))
sns.heatmap(df[num_cols_all].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

**Inference — correlation**
Linear correlations with `churn` are all weak (age is the strongest at ~0.29). That's expected — the biggest drivers we found above are categorical/threshold effects (active status, product count, age cliff), which a correlation coefficient can't capture. This points toward tree-based models, which handle non-linear and interaction effects natively.

## Key insights & preprocessing plan

**What the EDA suggests drives churn here (to be confirmed by the model):**
- Inactive membership, 3–4 products, age 40+, and being in Germany look like elevated-risk signals.
- Correlations are modest → relationships may be non-linear, so tree-based models (Random Forest, XGBoost) are worth testing against plain Logistic Regression; the comparison in section 10 decides.

**Preprocessing plan:**
- No missing values or duplicates to handle.
- **Outliers:** flagged with the z-score and IQR rules (section 4.1). Only `age` and `credit_score` have any, nearly all are *mild*, none is impossible, so **nothing is removed or capped**; section 10.1 confirms that capping would not help.
- Scale numeric columns (helps Logistic Regression; harmless for tree models).
- One-hot encode `country` and `gender`; 0/1 columns stay as one column.
- Engineer `age_group` (bins matching the observed churn jump) and a `zero_balance` flag (marks the spike at 0), and validate both with cross-validation before keeping them, not just because they seem reasonable.

## 7. Train / validation / test split

In [ ]:
features = ["credit_score", "country", "gender", "age", "tenure",
            "balance", "products", "credit_card", "active", "salary"]

X = df[features]
y = df[TARGET_COL]

# 70% train, 15% validation (for threshold tuning), 15% test (touched once, at the end)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print("Train:", len(X_train), " Val:", len(X_val), " Test:", len(X_test))
print("Churn rate — train:", round(y_train.mean(), 3),
      " val:", round(y_val.mean(), 3),
      " test:", round(y_test.mean(), 3))

Stratified splitting keeps the ~20% churn rate consistent across all three sets, so none of them accidentally becomes easier or harder than the others.

## 8. Feature engineering — then check if it helps

In [ ]:
ENGINEERED = ["zero_balance", "age_group"]

def add_features(data):
    out = data.copy()
    out["zero_balance"] = (out["balance"] == 0).astype(int)
    out["age_group"] = pd.cut(
        out["age"], bins=[0, 30, 40, 50, 60, 100], labels=[0, 1, 2, 3, 4]
    ).astype(int)
    return out

X_train_fe = add_features(X_train)
X_val_fe = add_features(X_val)
X_test_fe = add_features(X_test)

num_cols_model = ["credit_score", "age", "tenure", "balance", "products", "salary"]
num_cols_fe = num_cols_model + (ENGINEERED if USE_ENGINEERED_FEATURES else [])
cat_cols_model = ["country", "gender", "credit_card", "active"]

def make_preprocess(num_cols):
    # drop="if_binary": a 0/1 column becomes ONE column instead of two mirror-image ones
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), cat_cols_model),
    ])

preprocess = make_preprocess(num_cols_fe)

# Does adding the two features help, or just add noise?  Check on TWO model types:
# a linear model (which needs help with non-linear patterns) and a forest (which doesn't).
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

check_models = {
    "Logistic Regression": lambda: LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": lambda: RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced",
                                                     random_state=RANDOM_STATE, n_jobs=-1),
}
fe_rows = []
for mname, make_clf in check_models.items():
    for label, num_cols, data in [("without", num_cols_model, X_train),
                                  ("with", num_cols_model + ENGINEERED, X_train_fe)]:
        pipe = Pipeline([("prep", make_preprocess(num_cols)), ("model", make_clf())])
        s = cross_validate(pipe, data, y_train, cv=cv, scoring=["f1", "roc_auc"])
        fe_rows.append({"model": mname, "engineered features": label,
                        "f1": s["test_f1"].mean(), "roc_auc": s["test_roc_auc"].mean()})

with pd.option_context("display.float_format", "{:.4f}".format):
    display(pd.DataFrame(fe_rows))
print("Training with engineered features:", USE_ENGINEERED_FEATURES)

**Inference — feature engineering**
Read the table above: compare each model's F1 and ROC-AUC *with* vs *without* the two features. We keep `zero_balance` and `age_group` only if they help or at least do not hurt on both models. They are motivated by the EDA (the spike of balances at 0, the jump in churn after 40), but the cross-validation decides, not the story. Tree models can find such splits on their own, so little or no gain for the forest is normal; the gain, if any, shows mostly on the linear model.

To train without them, set `USE_ENGINEERED_FEATURES = False` in the config cell and re-run.

## 9. Baseline model

In [ ]:
baseline = Pipeline([("prep", preprocess),
                      ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
baseline.fit(X_train_fe, y_train)

proba = baseline.predict_proba(X_val_fe)[:, 1]
pred = (proba >= 0.5).astype(int)

print("Baseline (Logistic Regression, default threshold 0.5) on validation:")
print(f"  Accuracy : {accuracy_score(y_val, pred):.4f}")
print(f"  Precision: {precision_score(y_val, pred):.4f}")
print(f"  Recall   : {recall_score(y_val, pred):.4f}")
print(f"  F1       : {f1_score(y_val, pred):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(y_val, proba):.4f}")

This is the number every later model needs to beat. A plain Logistic Regression with class weighting is a fair, honest baseline — not a strawman.

## 10. Compare models

In [ ]:
n_stay = (y_train == 0).sum()
n_churn = (y_train == 1).sum()
scale = n_stay / n_churn

candidates = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", max_depth=8, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                              scale_pos_weight=scale, eval_metric="logloss",
                              random_state=RANDOM_STATE, n_jobs=-1),
}

scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision", "f1": "f1",
           "recall": "recall", "precision": "precision"}

rows = []
for name, clf in candidates.items():
    pipe = Pipeline([("prep", preprocess), ("model", clf)])
    scores = cross_validate(pipe, X_train_fe, y_train, cv=cv, scoring=scoring)
    rows.append({
        "model": name,
        "roc_auc": scores["test_roc_auc"].mean(),
        "roc_auc_std": scores["test_roc_auc"].std(),
        "pr_auc": scores["test_pr_auc"].mean(),
        "f1": scores["test_f1"].mean(),
        "recall": scores["test_recall"].mean(),
        "precision": scores["test_precision"].mean(),
    })

results_df = pd.DataFrame(rows).sort_values(SELECT_METRIC, ascending=False).reset_index(drop=True)
best_name = results_df.loc[0, "model"]
print(f"Best model by CV {SELECT_METRIC}:", best_name)

# Differences smaller than the fold-to-fold spread are not real differences
spread = results_df.loc[0, "roc_auc_std"]
close = results_df[results_df[SELECT_METRIC] >= results_df.loc[0, SELECT_METRIC] - spread]["model"].tolist()
if len(close) > 1:
    print(f"Within one std of the best (treat as a tie): {close}")
with pd.option_context("display.float_format", "{:.4f}".format):
    display(results_df)

In [ ]:
results_df.set_index("model")[["roc_auc", "pr_auc", "f1", "recall", "precision"]].plot(kind="bar", figsize=(9, 4), rot=0)
plt.ylim(0, 1)
plt.title("Cross-validated scores by model (threshold 0.5 for f1 / recall / precision)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Why ROC-AUC picks the model.** F1, recall and precision all depend on a threshold (0.5 here), and 0.5 is not the threshold we will end up using. ROC-AUC measures how well the model *ranks* churners above non-churners, whatever the threshold. PR-AUC is shown beside it because with only ~20% positives it is more sensitive to the churners being found at the top of the list. The threshold is chosen separately in sections 12–13.

If two models are within one standard deviation of each other (the "tie" line printed above), the data cannot tell them apart. In that case prefer the simpler one, or the one that is easier to explain.

### 10.1 Sensitivity check — would capping the outliers change the result?

Section 4.1 argued that the outliers are real customers and should stay. The cleanest way to back that up is to **try treating them and see whether anything improves**: cap `age` and `credit_score` at the IQR fences, and compare the same three candidate models with and without the cap.

The fences are learned **inside each cross-validation fold, from the training part only**, then applied to the held-out part, so the test of capping does not leak information either.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class IQRCapper(BaseEstimator, TransformerMixin):
    """Clip chosen columns to  Q1 - 1.5*IQR  and  Q3 + 1.5*IQR.
    The fences are learned in fit() (training folds only) and reused in transform() (held-out fold)."""
    def __init__(self, cols):
        self.cols = cols
    def fit(self, X, y=None):
        q1, q3 = X[self.cols].quantile(0.25), X[self.cols].quantile(0.75)
        self.low_, self.high_ = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
        return self
    def transform(self, X):
        X = X.copy()
        X[self.cols] = X[self.cols].clip(lower=self.low_, upper=self.high_, axis=1)
        return X


In [ ]:
KEEP, CAP = "keep outliers (what we did)", "cap age & credit_score"

sens_rows = []
for name, clf in candidates.items():
    for version, steps in [(KEEP, [("prep", preprocess), ("model", clf)]),
                           (CAP,  [("cap", IQRCapper(["age", "credit_score"])), ("prep", preprocess), ("model", clf)])]:
        s = cross_validate(Pipeline(steps), X_train_fe, y_train, cv=cv, scoring="roc_auc")
        sens_rows.append({"model": name, "version": version,
                          "roc_auc": s["test_score"].mean(), "fold_std": s["test_score"].std()})
sens = pd.DataFrame(sens_rows)

wide = sens.pivot(index="model", columns="version", values="roc_auc")[[KEEP, CAP]]
wide["difference (cap − keep)"] = wide[CAP] - wide[KEEP]
wide["fold-to-fold std"] = sens[sens["version"] == KEEP].set_index("model")["fold_std"]
wide["verdict"] = np.where(wide["difference (cap − keep)"].abs() < wide["fold-to-fold std"],
                           "within noise", "real change")
with pd.option_context("display.float_format", "{:.4f}".format):
    display(wide)


**Reading the table.**
- **Random Forest and XGBoost:** capping changes ROC-AUC by about −0.001, far smaller than the fold-to-fold spread, so it is *within noise* (and slightly negative). A tree only asks "is age above this cut-off?", so an extreme age lands in the same top bucket whether or not it was capped; capping gives it nothing.
- **Logistic Regression is the exception:** capping lifts it by about 0.009, more than its fold-to-fold spread. A linear model multiplies the scaled value by one weight, so a few very large ages can pull that weight. Even so it stays well behind the trees (about 0.76 against 0.86), and it is the baseline, not the model we ship.
- **Conclusion:** capping would only help the model we do not use. We keep the raw values, which also avoids an extra preprocessing step that would have to be reproduced in the API and the dashboard. If a linear model ever won the comparison in section 10, revisit this.

*(Numbers above are from one run; they will move slightly if you change the random seed or library versions, but the pattern, trees unaffected and linear slightly helped, is what to check.)*


## 11. Tune the winning model

In [ ]:
param_grids = {
    "Random Forest": {
        "model__n_estimators": [200, 400],
        "model__max_depth": [8, 12, 16, None],
        "model__min_samples_leaf": [2, 3, 5, 10],
    },
    "XGBoost": {
        "model__n_estimators": [200, 400],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__subsample": [0.7, 0.9],
    },
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10, 100],
    },
}

model_lookup = {
    "Random Forest": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(scale_pos_weight=scale, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1),
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE),
}

base_pipe = Pipeline([("prep", preprocess), ("model", model_lookup[best_name])])

grid = GridSearchCV(base_pipe, param_grids[best_name], scoring=SELECT_METRIC, cv=cv, n_jobs=-1)
grid.fit(X_train_fe, y_train)

tuned = grid.best_estimator_
print("Tuned model:", best_name)
print("Best params:", grid.best_params_)
print(f"Best CV {SELECT_METRIC}:", round(grid.best_score_, 4))

# If the best value sits at the edge of its range, the true optimum may be outside the grid
for p, best_val in grid.best_params_.items():
    options = param_grids[best_name][p]
    if len(options) > 1 and best_val in (options[0], options[-1]):
        print(f"  note: {p} = {best_val} is at the edge of {options}; consider widening this range")

Note the grid is looked up **by `best_name`** — whichever model actually won the comparison above gets tuned, not a hardcoded guess. Tuning uses the same metric as the model comparison (`SELECT_METRIC`), so the two steps agree on what "better" means. If a printed note says a best value is at the edge of its range, widen that range in `param_grids` and re-run.

## 12. Pick a threshold — maximise F1

In [ ]:
proba_val = tuned.predict_proba(X_val_fe)[:, 1]

thr_grid = np.round(np.arange(0.10, 0.90, 0.01), 2)
rows = []
for t in thr_grid:
    pred = (proba_val >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
    })

thr_df = pd.DataFrame(rows)
best_row = thr_df.sort_values("f1", ascending=False).iloc[0]
f1_threshold = float(best_row["threshold"])

plt.plot(thr_df["threshold"], thr_df["precision"], label="Precision")
plt.plot(thr_df["threshold"], thr_df["recall"], label="Recall")
plt.plot(thr_df["threshold"], thr_df["f1"], label="F1")
plt.axvline(f1_threshold, color="black", linestyle="--", alpha=0.5)
plt.xlabel("Threshold")
plt.legend()
plt.title("Precision / Recall / F1 vs. threshold (validation)")
plt.tight_layout()
plt.show()

print("Best threshold (by F1):", f1_threshold)
print(best_row)

F1 weighs a missed churner and a false alarm as equally bad. In practice they rarely cost the same: a retention offer sent to someone who was not leaving is cheap, losing a customer is not. The next section picks a threshold from costs instead. The dataset has no cost information, so every number there is an **assumption** you must replace.

## 13. Pick a threshold — maximise net benefit (assumed costs)

For every threshold, flag the customers scored at or above it and compute

`net benefit = (churners caught) × SUCCESS_RATE × VALUE_PER_SAVE − (customers flagged) × OFFER_COST`

The three inputs are set in the config cell and are **assumptions, not facts about any bank**. Because they are guesses, the useful output here is not one threshold but the table at the bottom: how much the best threshold moves when the guesses change.

Why the curve is computed instead of using a formula: the textbook shortcut `threshold = cost / (success × value)` only holds when the scores are true probabilities, and these are not (see section 14).

In [ ]:
def net_benefit_curve(y_true, scores, thresholds, cost=OFFER_COST, value=VALUE_PER_SAVE, success=SUCCESS_RATE):
    y_arr = np.asarray(y_true)
    out = []
    for t in thresholds:
        flag = np.asarray(scores) >= t
        tp = int((flag & (y_arr == 1)).sum())
        out.append({"threshold": t, "flagged": int(flag.sum()), "true_positives": tp,
                    "net_benefit": tp * success * value - flag.sum() * cost})
    return pd.DataFrame(out)

nb_val = net_benefit_curve(y_val, proba_val, thr_grid)
cost_threshold = float(nb_val.loc[nb_val["net_benefit"].idxmax(), "threshold"])

plt.plot(nb_val["threshold"], nb_val["net_benefit"])
plt.axvline(cost_threshold, color="black", linestyle="--", alpha=0.5, label=f"best = {cost_threshold}")
plt.axvline(f1_threshold, color="tab:red", linestyle=":", alpha=0.7, label=f"F1 threshold = {f1_threshold}")
plt.xlabel("Threshold"); plt.ylabel("Net benefit (units, on validation set)")
plt.title("Net benefit vs. threshold — ASSUMED costs")
plt.legend(); plt.tight_layout(); plt.show()

# How sensitive is the answer to the assumptions?
sens = []
for value in [5, 10, 20]:
    for success in [0.2, 0.3, 0.5]:
        c = net_benefit_curve(y_val, proba_val, thr_grid, value=value, success=success)
        i = c["net_benefit"].idxmax()
        sens.append({"value_per_save": value, "success_rate": success,
                     "best_threshold": c.loc[i, "threshold"], "flagged": c.loc[i, "flagged"],
                     "net_benefit": c.loc[i, "net_benefit"]})
display(pd.DataFrame(sens))

best_threshold = f1_threshold if THRESHOLD_RULE == "f1" else cost_threshold
print(f"THRESHOLD_RULE = {THRESHOLD_RULE!r}  ->  using threshold {best_threshold}")

**Choosing.** If the sensitivity table shows the best threshold jumping around as the assumptions change, the cost-based number is only as good as your guesses; keep `THRESHOLD_RULE = "f1"` and say so. If you can get real figures (offer cost, customer lifetime value, campaign success rate), put them in the config cell, set `THRESHOLD_RULE = "cost"` and re-run. Everything downstream uses `best_threshold`.

## 14. Checks before the final test: overfitting and calibration

In [ ]:
def score_block(model, Xf, yt, thr):
    p = model.predict_proba(Xf)[:, 1]
    pr = (p >= thr).astype(int)
    return {"recall": recall_score(yt, pr), "precision": precision_score(yt, pr, zero_division=0),
            "f1": f1_score(yt, pr), "roc_auc": roc_auc_score(yt, p)}

gap = pd.DataFrame({"train": score_block(tuned, X_train_fe, y_train, best_threshold),
                    "validation": score_block(tuned, X_val_fe, y_val, best_threshold)})
gap["gap (train - val)"] = gap["train"] - gap["validation"]
with pd.option_context("display.float_format", "{:.3f}".format):
    display(gap)

In [ ]:
frac_pos, mean_pred = calibration_curve(y_val, proba_val, n_bins=10, strategy="quantile")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="perfectly calibrated")
plt.plot(mean_pred, frac_pos, marker="o", label="model")
plt.xlabel("Mean predicted score"); plt.ylabel("Actual churn rate in that group")
plt.title("Calibration (validation set)")
plt.legend(); plt.tight_layout(); plt.show()

print(f"Mean score on validation : {proba_val.mean():.3f}")
print(f"Actual churn rate        : {y_val.mean():.3f}")

**Reading these two checks**
- **Overfitting:** a large gap between train and validation (the last column) means the model memorised the training customers. A small gap means the validation numbers are a fair preview of the test numbers. The threshold was picked on validation, so the validation figures are slightly optimistic; the test set in the next section is the honest one.
- **Calibration:** `class_weight="balanced"` (and `scale_pos_weight`) deliberately pushes scores up so churners are not ignored. If the mean score is well above the actual churn rate, as it usually is here, the output is **not a literal probability**. It is a *risk score*: good for ranking customers and for comparing against a threshold, wrong for statements like "this customer has a 70% chance of leaving". Use the words "risk score" in anything you show to stakeholders.

## 15. Final test (only once)

In [ ]:
X_final = pd.concat([X_train_fe, X_val_fe])
y_final = pd.concat([y_train, y_val])
scale_final = (y_final == 0).sum() / (y_final == 1).sum()

best_params = {k.replace("model__", ""): v for k, v in grid.best_params_.items()}

# Only build the classifier for the model that actually won — each model type
# takes different keyword arguments, so building all three eagerly would crash
# on whichever one didn't win.
if best_name == "Random Forest":
    final_clf = RandomForestClassifier(**best_params, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
elif best_name == "XGBoost":
    final_clf = XGBClassifier(**best_params, scale_pos_weight=scale_final, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)
elif best_name == "Logistic Regression":
    final_clf = LogisticRegression(**best_params, class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
else:
    raise ValueError(f"Unhandled model name: {best_name}")

final_model = Pipeline([("prep", preprocess), ("model", final_clf)])
final_model.fit(X_final, y_final)

proba_test = final_model.predict_proba(X_test_fe)[:, 1]
pred_test = (proba_test >= best_threshold).astype(int)

test_scores = {
    "accuracy": round(accuracy_score(y_test, pred_test), 4),
    "precision": round(precision_score(y_test, pred_test), 4),
    "recall": round(recall_score(y_test, pred_test), 4),
    "f1": round(f1_score(y_test, pred_test), 4),
    "roc_auc": round(roc_auc_score(y_test, proba_test), 4),
    "pr_auc": round(average_precision_score(y_test, proba_test), 4),
}
print("FINAL TEST RESULTS")
for k, v in test_scores.items():
    print(f"  {k:10s}: {v}")

In [ ]:
cm = confusion_matrix(y_test, pred_test)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred Stay", "Pred Churn"],
            yticklabels=["Actual Stay", "Actual Churn"])
plt.title(f"Confusion matrix (threshold={best_threshold})")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

print(classification_report(y_test, pred_test, target_names=["Stay", "Churn"]))

**How much can we trust these numbers?** The test set has only about 1,500 customers and roughly 300 churners, so every metric above carries sampling noise. Resampling the test set 1,000 times gives a 95% range for each one.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
y_arr = np.asarray(y_test)
n_test = len(y_arr)
boot = {"recall": [], "precision": [], "f1": [], "roc_auc": []}
for _ in range(1000):
    i = rng.integers(0, n_test, n_test)
    yt, ps = y_arr[i], proba_test[i]
    if yt.sum() in (0, n_test):
        continue
    pr = (ps >= best_threshold).astype(int)
    boot["recall"].append(recall_score(yt, pr))
    boot["precision"].append(precision_score(yt, pr, zero_division=0))
    boot["f1"].append(f1_score(yt, pr))
    boot["roc_auc"].append(roc_auc_score(yt, ps))

ci = pd.DataFrame({
    "point estimate": [test_scores[k] for k in boot],
    "95% low": [np.percentile(v, 2.5) for v in boot.values()],
    "95% high": [np.percentile(v, 97.5) for v in boot.values()],
}, index=list(boot))
with pd.option_context("display.float_format", "{:.3f}".format):
    display(ci)

The model is trained on train+validation combined (more data → a slightly stronger final model) and evaluated **exactly once** on the test set — this is the only honest way to report a number, since re-checking the test set repeatedly and picking the best run would silently overfit to it. One small caveat: the threshold was chosen using the model trained on train only, then applied to this refit model, so the two are very close but not identical.

When you quote results, quote the range from the table above, not just the single number.

## 16. Feature importance

In [ ]:
ohe = final_model.named_steps["prep"].named_transformers_["cat"]
feat_names = num_cols_fe + list(ohe.get_feature_names_out(cat_cols_model))

if hasattr(final_model.named_steps["model"], "feature_importances_"):
    importance = pd.Series(
        final_model.named_steps["model"].feature_importances_, index=feat_names
    ).sort_values(ascending=True)

    importance.tail(10).plot(kind="barh", color="steelblue")
    plt.title(f"Top features ({best_name})")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()

    print(importance.sort_values(ascending=False).head(10))
else:
    coefs = pd.Series(
        final_model.named_steps["model"].coef_[0], index=feat_names
    ).sort_values(key=abs, ascending=False)
    print(coefs.head(10))

**Main drivers:** compare the chart above with the EDA inferences. If the same features (activity, age, number of products, country) sit near the top, the model learned what the EDA showed, which is a useful sanity check that nothing leaked upstream. Note that built-in importance only says *how much* a feature is used, not *which direction* it pushes a customer, and it can favour features with many distinct values. SHAP in the next section fixes both.

## 17. Explaining predictions with SHAP

In [ ]:
import shap

prep_fit = final_model.named_steps["prep"]
clf = final_model.named_steps["model"]
clean_names = [n.split("__", 1)[-1] for n in prep_fit.get_feature_names_out()]

# Engineered columns belong to the original feature they were built from
GROUP_OF = {"age_group": "age", "zero_balance": "balance"}

def parent_feature(name):
    for c in cat_cols_model:                    # one-hot columns look like "country_Germany"
        if name == c or name.startswith(c + "_"):
            return c
    return GROUP_OF.get(name, name)

def to_dense(a):
    return a.toarray() if hasattr(a, "toarray") else np.asarray(a)

X_bg = to_dense(prep_fit.transform(X_train_fe))
if hasattr(clf, "feature_importances_"):
    explainer = shap.TreeExplainer(clf)
else:
    explainer = shap.LinearExplainer(clf, X_bg)

def shap_values_for(Xd):
    kwargs = {"check_additivity": False} if hasattr(clf, "feature_importances_") else {}
    sv = explainer.shap_values(Xd, **kwargs)
    if isinstance(sv, list):          # older shap: one array per class
        sv = sv[1]
    elif np.ndim(sv) == 3:            # newer shap: (rows, features, classes)
        sv = sv[:, :, 1]
    return np.asarray(sv)

def grouped_shap(frame_fe):
    """SHAP values per ORIGINAL feature (one-hot columns and engineered columns are summed back)."""
    Xd = to_dense(prep_fit.transform(frame_fe))
    sv = shap_values_for(Xd)
    return pd.DataFrame(sv, columns=clean_names, index=frame_fe.index).T.groupby(parent_feature).sum().T

shap_test = grouped_shap(X_test_fe)
shap_rank = shap_test.abs().mean().sort_values(ascending=True)

shap_rank.plot(kind="barh", color="steelblue")
plt.title(f"Mean |SHAP| per feature — test set ({best_name})")
plt.xlabel("Average effect on the score (model units: log-odds for XGBoost / logistic, probability for forest)")
plt.tight_layout(); plt.show()
print(shap_rank.sort_values(ascending=False).round(4))

In [ ]:
# Direction and spread: each dot is one customer; right = pushes the score up (towards churn)
Xd_test = to_dense(prep_fit.transform(X_test_fe))
shap.summary_plot(shap_values_for(Xd_test), Xd_test, feature_names=clean_names, show=False)
plt.tight_layout(); plt.show()

**How to read this.** The bar chart ranks the original features (a feature that was one-hot encoded or engineered is added back together) by how much it moves the score. The dot plot shows direction: for a feature, are the high values (red) on the right (raising churn risk) or the left? Check each against the EDA. Agreement is reassuring; disagreement is worth investigating before you present the model.

SHAP describes what the **model** relies on, not what *causes* customers to leave. "Inactive customers are scored higher" is a statement about the model and the data, not proof that re-activating them would stop them leaving.

## 18. Save the model & a simple prediction function

In [ ]:
joblib.dump(
    {"model": final_model, "threshold": best_threshold, "model_name": best_name, "test_scores": test_scores},
    "churn_model_notebook.joblib",
)
print("Saved: churn_model_notebook.joblib")

In [ ]:
def predict_churn(customer: dict) -> dict:
    """
    customer: dict with keys credit_score, country, gender, age, tenure,
              balance, products, credit_card, active, salary
    Returns the model's risk score and a yes/no call using the chosen threshold.

    NOTE: the score is NOT a calibrated probability (the model uses class weights, see section 14).
    The key is still called "churn_probability" so existing callers (e.g. the API) keep working;
    treat it as a risk score for ranking customers.
    """
    saved = joblib.load("churn_model_notebook.joblib")
    model, threshold = saved["model"], saved["threshold"]

    row = pd.DataFrame([customer])
    row = add_features(row)

    proba = model.predict_proba(row)[:, 1][0]
    return {
        "churn_probability": round(float(proba), 4),
        "will_churn": bool(proba >= threshold),
        "threshold_used": threshold,
    }

# Example
example_customer = {
    "credit_score": 600, "country": "Germany", "gender": "Male", "age": 45,
    "tenure": 3, "balance": 120000, "products": 1, "credit_card": 1,
    "active": 0, "salary": 55000,
}
predict_churn(example_customer)

## 19. Export the tables for the Power BI dashboard

Everything the dashboard needs comes from the objects above, so its numbers always match this notebook. Files go to `OUT_DIR`:

| File | One row per | Used for |
|---|---|---|
| `scored_customers.csv` | customer (all 10,000) | churn analysis page, call list, segments |
| `model_metrics.csv` | model / stage | model performance page |
| `confusion_test.csv` | confusion-matrix cell | model performance page |
| `roc_points.csv` | ROC curve point | model performance page |
| `threshold_curve.csv` | threshold | model performance page |
| `shap_importance.csv` | feature | churn analysis page |
| `net_benefit.csv` | assumption set × threshold | retention page (what-if) |

**How customers are scored fairly:** every customer's `risk_score` comes from a model that did **not** see that customer during training (5-fold out-of-fold predictions). Scoring customers with the model that was trained on them would make the model look far better than it is. `top_driver` is each customer's largest positive SHAP contribution from the final model (it is an explanation, not a prediction).

`risk_score` is a score, not a probability (section 14).

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
X_all_fe = add_features(X)

# ---- 1. out-of-fold risk score for every customer -------------------------------------------
df_out = df.copy()
df_out["risk_score"] = cross_val_predict(clone(final_model), X_all_fe, y, cv=cv, method="predict_proba")[:, 1]
df_out["flagged"] = (df_out["risk_score"] >= best_threshold).astype(int)
df_out["outcome"] = np.select(
    [(df_out["flagged"] == 1) & (df_out[TARGET_COL] == 1),
     (df_out["flagged"] == 1) & (df_out[TARGET_COL] == 0),
     (df_out["flagged"] == 0) & (df_out[TARGET_COL] == 1)],
    ["TP", "FP", "FN"], default="TN")

# ---- 2. bands for slicing --------------------------------------------------------------------
df_out["age_band"] = pd.cut(df_out["age"], [0, 29, 39, 49, 59, 200],
                            labels=["Under 30", "30s", "40s", "50s", "60+"]).astype(str)
mid_balance = df_out.loc[df_out["balance"] > 0, "balance"].median()
df_out["balance_band"] = np.select([df_out["balance"] == 0, df_out["balance"] <= mid_balance],
                                   ["Zero", "Low"], default="High")
df_out["risk_band"] = pd.qcut(df_out["risk_score"].rank(method="first"), 5,
                              labels=["Lowest", "Low", "Medium", "High", "Highest"]).astype(str)
df_out["segment"] = df_out["country"] + " | " + np.where(df_out["active"] == 1, "active", "inactive")

# ---- 3. each customer's biggest risk driver --------------------------------------------------
g_all = grouped_shap(X_all_fe)
df_out["top_driver"] = np.where(g_all.max(axis=1) > 0, g_all.idxmax(axis=1), "No risk driver")

# ---- 4. add identifiers back, restore the original column names -----------------------------
inverse = {v: k for k, v in RENAME.items()}
scored = ids.join(df_out).rename(columns=inverse)
cols = ["CustomerId", "Surname", "CreditScore", "Geography", "Gender", "Age", "Tenure", "Balance",
        "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary", "Exited",
        "risk_score", "flagged", "outcome", "risk_band", "age_band", "balance_band", "segment", "top_driver"]
scored[cols].round({"risk_score": 4}).to_csv(f"{OUT_DIR}/scored_customers.csv", index=False)

# ---- 5. model metrics (cross-validation rows + final test row with confidence range) --------
cv_rows = results_df.copy()
cv_rows.insert(1, "stage", "cross-validation (train, threshold 0.5)")
cv_rows["threshold"] = 0.5
final_row = {"model": best_name, "stage": "final (test split)", "threshold": best_threshold, **test_scores}
for m in ci.index:
    final_row[f"{m}_ci_low"] = ci.loc[m, "95% low"]
    final_row[f"{m}_ci_high"] = ci.loc[m, "95% high"]
pd.concat([cv_rows, pd.DataFrame([final_row])], ignore_index=True).round(4) \
  .to_csv(f"{OUT_DIR}/model_metrics.csv", index=False)

# ---- 6. confusion matrix, ROC points, threshold curve (all on the untouched test split) ------
cm = confusion_matrix(y_test, pred_test)           # [[TN, FP], [FN, TP]]
pd.DataFrame([
    {"actual": "Stayed", "predicted": "Stays",  "cell": "TN", "count": cm[0, 0], "share_of_actual": cm[0, 0] / cm[0].sum()},
    {"actual": "Stayed", "predicted": "Leaves", "cell": "FP", "count": cm[0, 1], "share_of_actual": cm[0, 1] / cm[0].sum()},
    {"actual": "Left",   "predicted": "Stays",  "cell": "FN", "count": cm[1, 0], "share_of_actual": cm[1, 0] / cm[1].sum()},
    {"actual": "Left",   "predicted": "Leaves", "cell": "TP", "count": cm[1, 1], "share_of_actual": cm[1, 1] / cm[1].sum()},
]).round(4).to_csv(f"{OUT_DIR}/confusion_test.csv", index=False)

fpr, tpr, thr = roc_curve(y_test, proba_test)
pd.DataFrame({"fpr": fpr, "tpr": tpr, "threshold": np.clip(thr, 0, 1)}).round(5) \
  .to_csv(f"{OUT_DIR}/roc_points.csv", index=False)

pd.DataFrame({
    "threshold": thr_grid,
    "recall": [recall_score(y_test, proba_test >= t) for t in thr_grid],
    "precision": [precision_score(y_test, proba_test >= t, zero_division=0) for t in thr_grid],
    "f1": [f1_score(y_test, proba_test >= t, zero_division=0) for t in thr_grid],
}).round(4).to_csv(f"{OUT_DIR}/threshold_curve.csv", index=False)

# ---- 7. SHAP importance (original features, test split) --------------------------------------
imp = shap_test.abs().mean().sort_values(ascending=False).reset_index()
imp.columns = ["feature", "mean_abs_shap"]
imp["rank"] = imp.index + 1
imp.round(5).to_csv(f"{OUT_DIR}/shap_importance.csv", index=False)

# ---- 8. net benefit under several ASSUMED cost sets, using all out-of-fold scores ------------
ts = np.round(np.arange(0.05, 0.96, 0.01), 2)
rows = []
for value in [5, 10, 20]:
    for success in [0.2, 0.3, 0.5]:
        c = net_benefit_curve(df_out[TARGET_COL], df_out["risk_score"], ts, cost=OFFER_COST, value=value, success=success)
        best_i = int(c["net_benefit"].idxmax())
        for i, r in c.iterrows():
            rows.append({"offer_cost": OFFER_COST, "value_per_save": value, "success_rate": success,
                         "threshold": r["threshold"], "flagged": int(r["flagged"]),
                         "true_positives": int(r["true_positives"]),
                         "net_benefit": round(float(r["net_benefit"]), 2), "is_best": int(i == best_i)})
pd.DataFrame(rows).to_csv(f"{OUT_DIR}/net_benefit.csv", index=False)

# ---- reconcile -------------------------------------------------------------------------------
print("Files in", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))
print("Customers:", len(scored), "| churners:", int(scored["Exited"].sum()),
      "| flagged:", int(scored["flagged"].sum()))
print("Out-of-fold recall at the chosen threshold:",
      round(recall_score(df_out[TARGET_COL], df_out["flagged"]), 3),
      "| precision:", round(precision_score(df_out[TARGET_COL], df_out["flagged"]), 3))

**Sanity check on the export.** The out-of-fold recall and precision printed above should be close to the test-set numbers in section 15 (they use different customers, so they will not match exactly). If they are far apart, something is wrong with the scoring and the dashboard should not be built on these files yet.

## Summary

### Data quality
- 10,000 rows, 0 missing, 0 duplicates — clean dataset (see section 3 output).

### EDA inferences
- **Inactive membership**, **3–4 products**, **age 40+**, and **Germany** are the clearest churn-risk signals in the plots.
- Correlations with churn are weak on their own → relationships are non-linear, which favours tree-based models; the CV comparison confirms or rejects that.

### Preprocessing
- Scaled numeric features, one-hot encoded categoricals.
- Outliers checked with z-score and IQR rules (section 4.1) and a capping experiment (section 10.1); deliberately left untouched.
- Added `zero_balance` and `age_group`, checked with cross-validation on two model types.

### Modeling
Baseline (Logistic Regression) → 3-model CV comparison on ROC-AUC → winner tuned by grid search → threshold chosen on validation (F1 or assumed-cost net benefit) → overfitting and calibration checks → evaluated once on held-out test with bootstrap ranges.

### What to report
Take the final numbers from the section 15 output and the confidence-range table (do not copy numbers from older runs or other write-ups). Call the output a **risk score**, not a probability. All cost figures are assumptions until replaced with real ones.

### Explanation
Feature importance and SHAP (sections 16–17) show what the model relies on; compare with the EDA.

### Outputs
`churn_model_notebook.joblib` (model + threshold; the app's own model is trained by `python -m churn.train`) and the CSVs in `dashboard/data/` for the Power BI dashboard.